# Experiment 001 — Typo Robustness: Colab Pilot

Full pilot pipeline in seven cells:
**clone → install → build items → annotate → generate → score → analyze + download**

Uses `qwen_1b5_pilot` (Qwen2.5-1.5B-Instruct, ungated, no HF auth needed) on a Colab T4.
The pilot config (`configs/pilot.yaml`) sets `is_confirmatory: false`, so pinned model
revisions are not required.

**Before starting:** Runtime → Change runtime type → T4 GPU.

### Cell 1 — Clone repo and install CPU dependencies

In [ ]:
import subprocess, sys

subprocess.run(
    ["git", "clone", "https://github.com/natSegOS/glamor-research-onboarding.git"],
    check=True)

%cd glamor-research-onboarding/001_typo_robustness

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"], check=True)
print("CPU deps installed")

### Cell 2 — Install GPU stack and verify CUDA

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-gpu.txt", "-q"], check=True)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU found — check Runtime > Change runtime type > T4 GPU")

### Cell 3 — Fetch and annotate the pilot item subset

Downloads 100 items per dataset from HuggingFace, then annotates each item with its
frozen key-term set K_P(x) using spaCy (design/04 §4.6). The spaCy model is downloaded
automatically on first run. Takes ~2–3 minutes on Colab.

In [ ]:
import subprocess, sys

# Fetch 100 items per dataset from HuggingFace and write pinned JSONL files.
subprocess.run([
    sys.executable, "tools/build_task_items.py",
    "--reasoning-items", "100",
    "--mcq-items", "100",
    "--seed", "1729",
    "--output-directory", "data/items",
], check=True)

# Annotate with frozen K_P(x) key terms.
# Downloads the spaCy model automatically if not already installed.
subprocess.run([
    sys.executable, "tools/build_annotated_dataset.py",
    "--model-name", "en_core_web_sm",
    "--items-dir", "data/items",
    "--force",
], check=True)

print("items ready in data/items/")

### Cell 4 — Build the English word dictionary

Fetches a pinned wordfreq English vocabulary and writes it to `data/wordlists/en_us_pinned.txt`.
This is the predicate that separates nonword typos (Regime A) from real-word shifts (Regime B).

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, "tools/build_dictionary.py",
], check=True)

print("dictionary ready")

### Cell 5 — Generate pilot outputs

Runs all conditions in `configs/pilot.yaml` (Regimes A, B, C) against `qwen_1b5_pilot`
(Qwen2.5-1.5B-Instruct). The runner is idempotent — if the cell is interrupted, re-run it
and it resumes from where it stopped. Writes to `results/pilot/pilot_generations.jsonl`.

Expected time: ~30–60 min on a T4 depending on condition count and token budget.

In [ ]:
import subprocess, sys

try:
    result = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True)
    git_commit = result.stdout.strip() or "unpinned"

    subprocess.run([
        sys.executable, "tools/run_generation.py",
        "--config", "configs/pilot.yaml",
        "--model", "qwen_1b5_pilot",
        "--output-directory", "results/pilot",
        "--dictionary", "data/wordlists/en_us_pinned.txt",
        "--git-commit", git_commit,
    ], check=True, capture_output=True, text=True)
    
    print("generation done — results/pilot/pilot_generations.jsonl")

except subprocess.CalledProcessError as e:
    print(f"\n[ERROR] Script crashed!")
    print(f"Exit Code: {e.returncode}")
    print(f"--- Standard Error (stderr) --- \n{e.stderr}")
    print(f"--- Standard Output (stdout) --- \n{e.stdout}")

### Cell 6 — Score generations

Applies the formal four-way parse-status classifier (design/04 §4.5) using spaCy dependency
parsing. Overwrites the inline parse_status with the research-grade linguistic classification.
Writes to `results/pilot/pilot_scored.jsonl`.

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, "tools/score_generations.py",
    "--input-path",  "results/pilot/pilot_generations.jsonl",
    "--output-path", "results/pilot/pilot_scored.jsonl",
    "--model-name",  "en_core_web_sm",
], check=True)

print("scoring done — results/pilot/pilot_scored.jsonl")

### Cell 7 — Analyze and download results

Produces the per-cell summary table, figures, and the HTML drill-down report.
Then downloads everything as a zip.

**Key pilot decisions to make from these outputs (design/11 §11.2):**
- Discordant rate per cell → sets N for the main run (Connor 1987; see design/06 §6.3)
- Which Regime B lever to keep (`regime_b_real_word` vs `regime_b_real_word_wide`)
- `max_new_tokens` → set to 99th percentile of clean-correct generation lengths
- Clean accuracy A₀ → validate against expected GSM-Symbolic / MMLU bands

In [ ]:
import subprocess, sys, zipfile, pathlib

# Cell summary table + statistical models + figures.
subprocess.run([
    sys.executable, "tools/run_analysis.py",
    "--generations",       "results/pilot/pilot_scored.jsonl",
    "--output-directory",  "analysis/pilot",
], check=True)

# Self-contained HTML drill-down report (global stats + per-item diff view).
subprocess.run([
    sys.executable, "tools/build_report.py",
    "--generations", "results/pilot/pilot_scored.jsonl",
    "--output",      "results/pilot/report.html",
], check=True)

# Zip everything and download.
zip_path = pathlib.Path("pilot_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in pathlib.Path("analysis/pilot").rglob("*"):
        zf.write(path)
    zf.write("results/pilot/report.html")
    zf.write("results/pilot/pilot_scored.jsonl")

try:
    from google.colab import files
    files.download(str(zip_path))
    print("downloaded pilot_results.zip")
except ImportError:
    print(f"results at {zip_path.resolve()}")